## タイトル

作成者 牧野航大



**調べたいこと**

- 緯度が反対になると季節が逆になるというが、本当に日本の夏のような暑さなのだろうか？
- 日本の今日の最高気温
- 日本と緯度が反対の位置の今日の最高気温
- 緯度も経度も真反対の今日の最高気温

In [5]:

import json
import urllib.request
import urllib.parse

# 東京の座標
tokyo_lat = 35.6762
tokyo_lon = 139.6503

# 比較する3地点
locations = {
    "東京": (tokyo_lat, tokyo_lon),
    "緯度だけ反対の地点": (-tokyo_lat, tokyo_lon),
    "地球の真反対の地点": (
        -tokyo_lat,
        ((tokyo_lon + 180) % 360 + 180) % 360 - 180
    )
}

def get_max_temperature(latitude, longitude):
    # Open-Meteo APIのパラメータ
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max",
        "forecast_days": 1,
        "timezone": "auto"
    }

    url = (
        "https://api.open-meteo.com/v1/forecast?"
        + urllib.parse.urlencode(params)
    )

    # APIからデータを取得
    with urllib.request.urlopen(url, timeout=20) as response:
        data = json.loads(response.read().decode("utf-8"))

    date = data["daily"]["time"][0]
    max_temp = data["daily"]["temperature_2m_max"][0]

    return date, max_temp


# 3地点の気温を取得して表示
results = {}

for name, (lat, lon) in locations.items():
    print(f"\n{name}のデータを取得中...")

    try:
        date, temp = get_max_temperature(lat, lon)
        results[name] = temp

        print(f"緯度：{lat:.4f}")
        print(f"経度：{lon:.4f}")
        print(f"予報日：{date}")

        if temp is not None:
            print(f"最高気温：{temp} ℃")
        else:
            print("最高気温：データなし")

    except Exception as error:
        results[name] = None
        print(f"取得失敗：{error}")


# 気温を比較
print("\n========== 最高気温の比較 ==========")

for name, temp in results.items():
    if temp is not None:
        print(f"{name}：{temp} ℃")

tokyo_temp = results["東京"]

if tokyo_temp is not None:
    print("\n東京との気温差")

    for name, temp in results.items():
        if name != "東京" and temp is not None:
            difference = temp - tokyo_temp
            print(f"{name}：{difference:+.1f} ℃")



東京のデータを取得中...
緯度：35.6762
経度：139.6503
予報日：2026-10-21
最高気温：28.7 ℃

緯度だけ反対の地点のデータを取得中...
緯度：-35.6762
経度：139.6503
予報日：2026-10-21
最高気温：26.5 ℃

地球の真反対の地点のデータを取得中...
緯度：-35.6762
経度：-40.3497
予報日：2026-10-21
最高気温：13.2 ℃

========== 最高気温の比較 ==========
東京：28.7 ℃
緯度だけ反対の地点：26.5 ℃
地球の真反対の地点：13.2 ℃

東京との気温差
緯度だけ反対の地点：-2.2 ℃
地球の真反対の地点：-15.5 ℃


###　実行時のセル出力

東京のデータを取得中...
緯度：35.6762
経度：139.6503
予報日：2026-10-21
最高気温：28.7 ℃

緯度だけ反対の地点のデータを取得中...
緯度：-35.6762
経度：139.6503
予報日：2026-10-21
最高気温：26.5 ℃

地球の真反対の地点のデータを取得中...
緯度：-35.6762
経度：-40.3497
予報日：2026-10-21
最高気温：13.2 ℃

========== 最高気温の比較 ==========
東京：28.7 ℃
緯度だけ反対の地点：26.5 ℃
地球の真反対の地点：13.2 ℃

東京との気温差
緯度だけ反対の地点：-2.2 ℃
地球の真反対の地点：-15.5 ℃

### わかったこと

- 気温では緯度の反対の地域では25度を超える初夏相当の気温だとわかった
- 日本の最高気温が高かった日なので比較としてはいまいちだった。そもそも現在の日本は秋で熱いとも寒いともどちらとも形容しにくい季節だった
- ↑より差がほどほどで収まる緯度だけ真反対の地域は適切な結果だったといえる
- しかし気球の真反対の地域が非常に気温が低かったのには緯度が反対の地域と比べて疑問となった
- 海の上だからかその他の理由だったのかは検証できなかった。